# 01 - Spectral Reliability Baseline, Original SIT Classes, Memory-Light Version

This notebook avoids the memory issue caused by storing spectral bands, normalized bands, and prototype bands inside large pandas DataFrames.

Key changes:

- `X` remains a NumPy `float32` array.
- `df` stores **metadata only**: sample index, label, field id, scene id, class support.
- Prototypes and pixel scores are computed class-by-class using NumPy arrays.
- `df_scores` is compact and contains only metadata + scores, not spectral bands.
- Original SIT `LIVELLO_4` labels are preserved without class merging.


In [1]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

print("Libraries loaded.")

Libraries loaded.


## 1. Configuration

Edit paths before running.


In [2]:
# =========================================================
# USER CONFIGURATION
# =========================================================


BASE_PATH = Path("/kaggle/input/datasets/andgu2026/prisma-data2/")
AUDIT_DIR = Path("/kaggle/input/datasets/andgu2026/prisma-data2/")

DATA_FILE = BASE_PATH / "pretraining_data.csv"
LABEL_FILE = BASE_PATH / "pretraining_data_y.csv"
FIELD_FILE = BASE_PATH / "pretraining_data_field_ids.csv"
SCENE_FILE = BASE_PATH / "pretraining_data_scene_ids.csv"

OUT_DIR = Path("/kaggle/working")
TABLE_DIR = OUT_DIR / "tables"
FIGURE_DIR = OUT_DIR / "figures"
REPORT_DIR = OUT_DIR / "reports"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)




# Optional consistency files, if available.
ORIGINAL_CODE_FILE = BASE_PATH / "pretraining_data_original_sit_codes.csv"
EXPECTED_Y_FILE = BASE_PATH / "pretraining_data_expected_y_from_field_id.csv"


SETTING = "full"  # "full" or "summer"

SUMMER_SCENES = [
    "Scena_1", "Scena_4", "Scena_7", "Scena_8", "Scena_9",
    "Scena_10", "Scena_14", "Scena_23", "Scena_25",
]

CLASSES_TO_REMOVE = []

# If memory is still limited, set this to e.g. 1000000 or 500000.
# None = use all pixels.
MAX_SAMPLES = None
SUBSAMPLE_SEED = 42

MIN_FIELD_PIXELS = 10
MIN_CLASS_PIXELS = 1000
MIN_CLASS_FIELDS = 30
MIN_CLASS_SCENES = 3

PRIORITY_RELIABILITY_Q = 0.10
UNCERTAIN_RELIABILITY_Q = 0.30
PRIORITY_OUTLIER_FRACTION_Q = 0.90

# Processing chunk size for within-class arrays.
CHUNK_SIZE = 250000

print("BASE_PATH:", BASE_PATH)
print("AUDIT_DIR:", AUDIT_DIR)
print("TABLE_DIR:", TABLE_DIR)

BASE_PATH: /kaggle/input/datasets/andgu2026/prisma-data2
AUDIT_DIR: /kaggle/input/datasets/andgu2026/prisma-data2
TABLE_DIR: /kaggle/working/tables


## 2. Class metadata

Add missing original SIT classes here if needed.


In [3]:
CLASS_INFO = {
    111:  ("Continuous urban fabric", "artificial"),
    112:  ("Discontinuous urban fabric", "artificial"),
    121:  ("Industrial or commercial units", "artificial"),
    122:  ("Road and rail networks", "artificial"),
    123:  ("Port areas", "artificial"),
    124:  ("Airports", "artificial"),
    131:  ("Mineral extraction sites", "artificial"),
    132:  ("Dump sites", "artificial"),
    133:  ("Construction sites", "artificial"),
    141:  ("Green urban areas", "semi_dynamic"),
    142:  ("Sport and leisure facilities", "semi_dynamic"),

    2111: ("Non-irrigated arable land", "dynamic_agricultural"),
    2112: ("Non-irrigated horticultural crops", "dynamic_agricultural"),
    2121: ("Irrigated arable land", "dynamic_agricultural"),
    2123: ("Irrigated horticultural crops", "dynamic_agricultural"),

    221:  ("Vineyards", "semi_dynamic"),
    222:  ("Fruit trees and minor fruit plantations", "semi_dynamic"),
    223:  ("Olive groves", "persistent"),
    224:  ("Other permanent crops", "semi_dynamic"),
    231:  ("Dense herbaceous cover", "semi_dynamic"),
    241:  ("Annual crops associated with permanent crops", "dynamic_agricultural"),
    242:  ("Complex cultivation patterns", "dynamic_agricultural"),
    243:  ("Agricultural areas with natural vegetation", "semi_dynamic"),
    244:  ("Agro-forestry areas", "semi_dynamic"),

    311:  ("Broad-leaved forest", "persistent"),
    312:  ("Coniferous forest", "persistent"),
    313:  ("Mixed forest", "persistent"),
    314:  ("Other wooded / forest class", "persistent"),

    321:  ("Natural grassland, pasture, and fallow land", "semi_dynamic"),
    322:  ("Shrubland", "persistent"),
    323:  ("Sclerophyllous vegetation", "persistent"),
    324:  ("Transitional woodland/shrub", "semi_dynamic"),
    3241: ("Natural recolonization areas", "semi_dynamic"),
    3242: ("Natural recolonization areas type 2", "semi_dynamic"),

    331:  ("Beaches, dunes, and sand", "persistent"),
    332:  ("Bare rock and outcrops", "persistent"),
    333:  ("Sparsely vegetated areas", "semi_dynamic"),
    334:  ("Burnt areas", "semi_dynamic"),

    411:  ("Inland marshes", "semi_dynamic"),
    421:  ("Salt marshes", "semi_dynamic"),
    511:  ("Water courses", "water"),
    512:  ("Water bodies", "water"),
    521:  ("Coastal lagoons", "water"),
    522:  ("Estuaries", "water"),
    523:  ("Sea and ocean", "water"),
}

CLASS_INFO_DF = pd.DataFrame(
    [{"class_label": int(k), "class_name": v[0], "persistence_group": v[1]} for k, v in CLASS_INFO.items()]
)

display(CLASS_INFO_DF.sort_values("class_label"))

,class_label,class_name,persistence_group
0,111,Continuous urban fabric,artificial
1,112,Discontinuous urban fabric,artificial
2,121,Industrial or commercial units,artificial
3,122,Road and rail networks,artificial
4,123,Port areas,artificial
5,124,Airports,artificial
6,131,Mineral extraction sites,artificial
7,132,Dump sites,artificial
8,133,Construction sites,artificial
9,141,Green urban areas,semi_dynamic


## 3. Utility functions

In [4]:
def read_vector_csv(path, dtype=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    v = pd.read_csv(path, header=None).iloc[:, 0].values
    if dtype is not None:
        v = v.astype(dtype)
    return v


def normalize_scene_id_array(scene_ids):
    raw = pd.Series(scene_ids).astype(str).str.strip()
    cleaned = raw.str.replace(r"[-\s]+", "_", regex=True)
    extracted = cleaned.str.extract(r"(?i)scena_?(\d+)", expand=False)
    out = cleaned.copy()
    mask = extracted.notna()
    out.loc[mask] = extracted.loc[mask].astype(int).map(lambda n: f"Scena_{n}")
    return out.values


def subsample_balanced_by_scene_indices(scene_ids, max_samples=None, seed=42):
    scene_ids = np.asarray(scene_ids)
    n_total = len(scene_ids)

    if max_samples is None or n_total <= max_samples:
        return np.arange(n_total, dtype=np.int64)

    rng = np.random.default_rng(seed)
    scenes = sorted(pd.Series(scene_ids).unique())
    base_per_scene = max_samples // len(scenes)
    remainder = max_samples % len(scenes)

    chosen = []
    for i, scene in enumerate(scenes):
        idx = np.where(scene_ids == scene)[0]
        n = base_per_scene + (1 if i < remainder else 0)
        n = min(n, len(idx))
        chosen.extend(rng.choice(idx, size=n, replace=False).tolist())

    chosen = np.array(sorted(set(chosen)), dtype=np.int64)

    if len(chosen) < max_samples:
        selected_mask = np.zeros(n_total, dtype=bool)
        selected_mask[chosen] = True
        remaining = np.where(~selected_mask)[0]
        n_extra = min(max_samples - len(chosen), len(remaining))
        if n_extra > 0:
            extra = rng.choice(remaining, size=n_extra, replace=False)
            chosen = np.array(sorted(np.concatenate([chosen, extra])), dtype=np.int64)

    return chosen


def l2_normalize_rows(A, eps=1e-12):
    A = np.asarray(A, dtype=np.float32)
    norm = np.linalg.norm(A, axis=1, keepdims=True).astype(np.float32)
    return A / np.maximum(norm, eps)


def robust_median_iqr(x):
    x = np.asarray(x, dtype=np.float32)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan, np.nan
    q25 = np.quantile(x, 0.25)
    q50 = np.quantile(x, 0.50)
    q75 = np.quantile(x, 0.75)
    return float(q25), float(q50), float(q75), float(q75 - q25)


def spectral_scores_to_proto(X_class, proto, chunk_size=250000):
    n = X_class.shape[0]
    sad = np.empty(n, dtype=np.float32)
    edist = np.empty(n, dtype=np.float32)

    proto = proto.astype(np.float32)
    proto_norm = max(float(np.linalg.norm(proto)), 1e-12)
    proto = proto / proto_norm

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        Xc = X_class[start:end].astype(np.float32, copy=False)
        Xn = l2_normalize_rows(Xc)

        dots = np.sum(Xn * proto[None, :], axis=1)
        dots = np.clip(dots, -1.0, 1.0)
        sad[start:end] = np.arccos(dots).astype(np.float32)
        edist[start:end] = np.linalg.norm(Xn - proto[None, :], axis=1).astype(np.float32)

    return sad, edist


print("Utility functions ready.")

Utility functions ready.


## 4. Load global dataset

In [5]:
for p in [DATA_FILE, LABEL_FILE, FIELD_FILE, SCENE_FILE]:
    if not p.exists():
        raise FileNotFoundError(f"Missing input file: {p}")

print("Loading X:", DATA_FILE)
X = pd.read_csv(DATA_FILE, header=None).values.astype(np.float32)

print("Loading y:", LABEL_FILE)
y = read_vector_csv(LABEL_FILE, dtype=np.int64)

print("Loading field ids:", FIELD_FILE)
field_ids = read_vector_csv(FIELD_FILE, dtype=np.int64)

print("Loading scene ids:", SCENE_FILE)
scene_ids = normalize_scene_id_array(read_vector_csv(SCENE_FILE))

if len(X) != len(y) or len(X) != len(field_ids) or len(X) != len(scene_ids):
    raise ValueError(f"Length mismatch: X={len(X)}, y={len(y)}, field_ids={len(field_ids)}, scene_ids={len(scene_ids)}")

INPUT_DIM = X.shape[1]

print("X shape:", X.shape)
print("X memory GB:", X.nbytes / 1e9)
print("Unique scenes:", sorted(pd.Series(scene_ids).unique()))
print("Unique classes:", sorted(pd.Series(y).unique()))

Loading X: /kaggle/input/datasets/andgu2026/prisma-data2/pretraining_data.csv
Loading y: /kaggle/input/datasets/andgu2026/prisma-data2/pretraining_data_y.csv
Loading field ids: /kaggle/input/datasets/andgu2026/prisma-data2/pretraining_data_field_ids.csv
Loading scene ids: /kaggle/input/datasets/andgu2026/prisma-data2/pretraining_data_scene_ids.csv
X shape: (2500000, 191)
X memory GB: 1.91
Unique scenes: ['Scena_1', 'Scena_10', 'Scena_11', 'Scena_12', 'Scena_13', 'Scena_14', 'Scena_15', 'Scena_16', 'Scena_17', 'Scena_18', 'Scena_19', 'Scena_2', 'Scena_20', 'Scena_21', 'Scena_22', 'Scena_23', 'Scena_24', 'Scena_25', 'Scena_3', 'Scena_4', 'Scena_5', 'Scena_6', 'Scena_7', 'Scena_8', 'Scena_9']
Unique classes: [np.int64(221), np.int64(222), np.int64(223), np.int64(224), np.int64(231), np.int64(241), np.int64(242), np.int64(243), np.int64(244), np.int64(311), np.int64(312), np.int64(313), np.int64(314), np.int64(321), np.int64(322), np.int64(323), np.int64(331), np.int64(332), np.int64(333),

In [6]:
# Optional consistency checks.
if ORIGINAL_CODE_FILE.exists():
    original_sit_codes = read_vector_csv(ORIGINAL_CODE_FILE, dtype=np.int64)
    if len(original_sit_codes) != len(y):
        raise ValueError("original_sit_codes length differs from y.")
    print(f"Mismatch y vs original_sit_codes: {np.sum(original_sit_codes != y)} / {len(y)}")

if EXPECTED_Y_FILE.exists():
    expected_y = read_vector_csv(EXPECTED_Y_FILE, dtype=np.int64)
    if len(expected_y) != len(y):
        raise ValueError("expected_y length differs from y.")
    print(f"Mismatch y vs expected_y_from_field_id: {np.sum(expected_y != y)} / {len(y)}")

## 5. Apply basic filters without creating a spectral DataFrame

In [7]:
# =========================================================
# BASIC FILTERS ON ARRAYS
# =========================================================

keep = np.ones(len(y), dtype=bool)

if len(CLASSES_TO_REMOVE) > 0:
    keep &= ~np.isin(y, np.array(CLASSES_TO_REMOVE, dtype=np.int64))

if SETTING == "summer":
    keep &= np.isin(scene_ids, np.array(SUMMER_SCENES, dtype=str))
elif SETTING == "full":
    pass
else:
    raise ValueError(f"Unknown SETTING={SETTING}")

print("Rows before filters:", len(y))
print("Rows after basic filters:", int(keep.sum()))

X = X[keep]
y = y[keep]
field_ids = field_ids[keep]
scene_ids = scene_ids[keep]

# Optional balanced-by-scene subsampling.
idx_sub = subsample_balanced_by_scene_indices(scene_ids, max_samples=MAX_SAMPLES, seed=SUBSAMPLE_SEED)

X = X[idx_sub]
y = y[idx_sub]
field_ids = field_ids[idx_sub]
scene_ids = scene_ids[idx_sub]

sample_index = np.arange(len(y), dtype=np.int64)

print("\nAfter basic filtering/subsampling:")
print("Rows:", len(y))
print("X shape:", X.shape)
print("X memory GB:", X.nbytes / 1e9)
print("Scenes:", pd.Series(scene_ids).nunique())
print("Classes:", sorted(pd.Series(y).unique()))
print("Field-scene units:", pd.DataFrame({"scene_id": scene_ids, "field_id": field_ids}).drop_duplicates().shape[0])

gc.collect()

Rows before filters: 2500000
Rows after basic filters: 2500000

After basic filtering/subsampling:
Rows: 2500000
X shape: (2500000, 191)
X memory GB: 1.91
Scenes: 25
Classes: [np.int64(221), np.int64(222), np.int64(223), np.int64(224), np.int64(231), np.int64(241), np.int64(242), np.int64(243), np.int64(244), np.int64(311), np.int64(312), np.int64(313), np.int64(314), np.int64(321), np.int64(322), np.int64(323), np.int64(331), np.int64(332), np.int64(333), np.int64(334), np.int64(2111), np.int64(2112), np.int64(2121), np.int64(2123), np.int64(3241), np.int64(3242)]
Field-scene units: 190914


47

## 6. Metadata DataFrame and class support

`df` contains metadata only, not spectral bands.


In [8]:
# =========================================================
# METADATA-ONLY DATAFRAME
# =========================================================

df = pd.DataFrame({
    "sample_index": sample_index,
    "class_label": y.astype(np.int64),
    "field_id": field_ids.astype(np.int64),
    "scene_id": scene_ids.astype(str),
})
df["setting"] = SETTING

# Create stable class_id.
unique_labels = sorted(df["class_label"].unique())
label_to_id = {label: i for i, label in enumerate(unique_labels)}
df["class_id"] = df["class_label"].map(label_to_id).astype(np.int64)

df = df.merge(CLASS_INFO_DF, on="class_label", how="left")

# Fallback for missing class info.
if df["class_name"].isna().any():
    missing = sorted(df.loc[df["class_name"].isna(), "class_label"].unique())
    print(f"WARNING: Missing CLASS_INFO for classes: {missing}")
    df["class_name"] = df["class_name"].fillna(df["class_label"].apply(lambda x: f"SIT class {int(x)}"))
    df["persistence_group"] = df["persistence_group"].fillna("unknown")

display(df.head())

,sample_index,class_label,field_id,scene_id,setting,class_id,class_name,persistence_group
0,0,312,16415,Scena_1,full,10,Coniferous forest,persistent
1,1,323,18474,Scena_1,full,15,Sclerophyllous vegetation,persistent
2,2,223,4962,Scena_1,full,2,Olive groves,persistent
3,3,221,3045,Scena_1,full,0,Vineyards,semi_dynamic
4,4,323,18296,Scena_1,full,15,Sclerophyllous vegetation,persistent


In [9]:
# =========================================================
# CLASS-LEVEL SUPPORT FILTER FOR ORIGINAL SIT LABELS
# =========================================================

field_support_tmp = (
    df.groupby(["scene_id", "field_id", "class_label"], dropna=False)
    .size()
    .reset_index(name="field_n_pixels")
)

class_support = (
    field_support_tmp
    .groupby("class_label", dropna=False)
    .agg(
        class_n_pixels=("field_n_pixels", "sum"),
        class_n_fields=("field_id", "count"),
        class_n_scenes=("scene_id", "nunique"),
    )
    .reset_index()
)

class_support["class_supported"] = (
    (class_support["class_n_pixels"] >= MIN_CLASS_PIXELS) &
    (class_support["class_n_fields"] >= MIN_CLASS_FIELDS) &
    (class_support["class_n_scenes"] >= MIN_CLASS_SCENES)
)

class_support = class_support.merge(CLASS_INFO_DF, on="class_label", how="left")
class_support["class_name"] = class_support["class_name"].fillna(class_support["class_label"].apply(lambda x: f"SIT class {int(x)}"))
class_support["persistence_group"] = class_support["persistence_group"].fillna("unknown")

df = df.merge(
    class_support[["class_label", "class_n_pixels", "class_n_fields", "class_n_scenes", "class_supported"]],
    on="class_label",
    how="left",
)

df["class_supported"] = df["class_supported"].fillna(False).astype(bool)

eval_mask = df["class_supported"].values

print("\nClass support:")
display(class_support.sort_values(["class_supported", "class_n_pixels"], ascending=[True, False]))

print("\nRows used for reliability estimation:", int(eval_mask.sum()))
print("Supported classes:", sorted(df.loc[df["class_supported"], "class_label"].unique()))
print("Unsupported classes:", sorted(df.loc[~df["class_supported"], "class_label"].unique()))


Class support:


,class_label,class_n_pixels,class_n_fields,class_n_scenes,class_supported,class_name,persistence_group
3,224,828,109,10,False,Other permanent crops,semi_dynamic
23,2123,610,161,21,False,Irrigated horticultural crops,dynamic_agricultural
8,244,257,60,3,False,Agro-forestry areas,semi_dynamic
19,334,154,25,4,False,Burnt areas,semi_dynamic
2,223,734113,44931,25,True,Olive groves,persistent
20,2111,675874,55696,25,True,Non-irrigated arable land,dynamic_agricultural
0,221,244804,27419,24,True,Vineyards,semi_dynamic
22,2121,218679,3008,16,True,Irrigated arable land,dynamic_agricultural
13,321,142643,18135,25,True,"Natural grassland, pasture, and fallow land",semi_dynamic
9,311,138395,4471,25,True,Broad-leaved forest,persistent



Rows used for reliability estimation: 2498151
Supported classes: [np.int64(221), np.int64(222), np.int64(223), np.int64(231), np.int64(241), np.int64(242), np.int64(243), np.int64(311), np.int64(312), np.int64(313), np.int64(314), np.int64(321), np.int64(322), np.int64(323), np.int64(331), np.int64(332), np.int64(333), np.int64(2111), np.int64(2112), np.int64(2121), np.int64(3241), np.int64(3242)]
Unsupported classes: [np.int64(224), np.int64(244), np.int64(334), np.int64(2123)]


## 7. Compute prototypes and compact pixel scores

This step processes one class at a time and avoids creating huge normalized/prototype columns.


In [10]:
# =========================================================
# CLASS PROTOTYPES AND PIXEL SCORES, MEMORY-LIGHT
# =========================================================

supported_classes = sorted(df.loc[df["class_supported"], "class_label"].unique())

score_parts = []
prototype_rows = []
dist_stats_rows = []

for class_label in supported_classes:
    print(f"Processing class {class_label}...")

    idx = np.where((y == class_label) & eval_mask)[0]
    Xc = X[idx].astype(np.float32, copy=False)

    # Class prototype: median of normalized spectra.
    # This stores only this class in memory.
    Xc_norm = l2_normalize_rows(Xc)
    proto = np.median(Xc_norm, axis=0).astype(np.float32)
    proto = proto / max(float(np.linalg.norm(proto)), 1e-12)

    proto_row = {"class_label": int(class_label)}
    for j, val in enumerate(proto):
        proto_row[f"proto_{j}"] = float(val)
    prototype_rows.append(proto_row)

    # Free normalized class matrix before score loop if possible.
    del Xc_norm
    gc.collect()

    sad, edist = spectral_scores_to_proto(Xc, proto, chunk_size=CHUNK_SIZE)

    sad_q25, sad_med, sad_q75, sad_iqr = robust_median_iqr(sad)
    ed_q25, ed_med, ed_q75, ed_iqr = robust_median_iqr(edist)

    dist_stats_rows.append({
        "class_label": int(class_label),
        "sad_q25_class": sad_q25,
        "sad_median_class": sad_med,
        "sad_q75_class": sad_q75,
        "sad_iqr_class": sad_iqr,
        "edist_q25_class": ed_q25,
        "edist_median_class": ed_med,
        "edist_q75_class": ed_q75,
        "edist_iqr_class": ed_iqr,
    })

    eps = 1e-12
    sad_z = (sad - sad_med) / max(sad_iqr, eps)
    ed_z = (edist - ed_med) / max(ed_iqr, eps)
    z = np.maximum(0, np.maximum(sad_z, ed_z))
    reliability = np.exp(-z).astype(np.float32)

    sad_thr = sad_q75 + 1.5 * sad_iqr
    ed_thr = ed_q75 + 1.5 * ed_iqr
    is_outlier = (sad > sad_thr) | (edist > ed_thr)

    part = pd.DataFrame({
        "sample_index": sample_index[idx],
        "scene_id": scene_ids[idx],
        "field_id": field_ids[idx],
        "class_label": y[idx],
        "sad": sad.astype(np.float32),
        "edist": edist.astype(np.float32),
        "sad_robust_z": sad_z.astype(np.float32),
        "edist_robust_z": ed_z.astype(np.float32),
        "reliability": reliability.astype(np.float32),
        "is_outlier": is_outlier.astype(bool),
    })

    score_parts.append(part)

    del Xc, sad, edist, sad_z, ed_z, z, reliability, is_outlier, part
    gc.collect()

prototypes = pd.DataFrame(prototype_rows)
dist_stats = pd.DataFrame(dist_stats_rows)
df_scores = pd.concat(score_parts, ignore_index=True)

df_scores = df_scores.merge(
    df[["sample_index", "class_name", "persistence_group", "setting"]],
    on="sample_index",
    how="left",
)

del score_parts
gc.collect()

print("df_scores:", df_scores.shape)
display(df_scores.head())

Processing class 221...
Processing class 222...
Processing class 223...
Processing class 231...
Processing class 241...
Processing class 242...
Processing class 243...
Processing class 311...
Processing class 312...
Processing class 313...
Processing class 314...
Processing class 321...
Processing class 322...
Processing class 323...
Processing class 331...
Processing class 332...
Processing class 333...
Processing class 2111...
Processing class 2112...
Processing class 2121...
Processing class 3241...
Processing class 3242...
df_scores: (2498151, 13)


,sample_index,scene_id,field_id,class_label,sad,edist,sad_robust_z,edist_robust_z,reliability,is_outlier,class_name,persistence_group,setting
0,3,Scena_1,3045,221,0.087090,0.087063,-0.539489,-0.540211,1.000000,False,Vineyards,semi_dynamic,full
1,9,Scena_1,6127,221,0.080760,0.080739,-0.596929,-0.597773,1.000000,False,Vineyards,semi_dynamic,full
2,122,Scena_1,3127,221,0.095869,0.095831,-0.459822,-0.460393,1.000000,False,Vineyards,semi_dynamic,full
3,125,Scena_1,3148,221,0.085463,0.085437,-0.554246,-0.555004,1.000000,False,Vineyards,semi_dynamic,full
4,161,Scena_1,9081,221,0.146918,0.146786,0.003429,0.003437,0.996569,False,Vineyards,semi_dynamic,full


## 8. Build field-level summary

In [11]:
# =========================================================
# FIELD BASE TABLE FROM COMPLETE METADATA
# =========================================================

field_base = (
    df.groupby(
        ["scene_id", "field_id", "class_label", "class_name", "persistence_group", "setting"],
        dropna=False,
    )
    .agg(
        n_pixels=("sample_index", "count"),
        class_supported=("class_supported", "first"),
        class_n_pixels=("class_n_pixels", "first"),
        class_n_fields=("class_n_fields", "first"),
        class_n_scenes=("class_n_scenes", "first"),
    )
    .reset_index()
)

print("field_base:", field_base.shape)
display(field_base.head())

field_base: (190914, 11)


,scene_id,field_id,class_label,class_name,persistence_group,setting,n_pixels,class_supported,class_n_pixels,class_n_fields,class_n_scenes
0,Scena_1,1,221,Vineyards,semi_dynamic,full,3,True,244804,27419,24
1,Scena_1,2,221,Vineyards,semi_dynamic,full,2,True,244804,27419,24
2,Scena_1,5,221,Vineyards,semi_dynamic,full,1,True,244804,27419,24
3,Scena_1,7,221,Vineyards,semi_dynamic,full,1,True,244804,27419,24
4,Scena_1,8,221,Vineyards,semi_dynamic,full,1,True,244804,27419,24


In [12]:
# =========================================================
# FIELD SCORES ONLY FROM SUPPORTED CLASSES
# =========================================================

field_scores = (
    df_scores.groupby(["scene_id", "field_id", "class_label"], dropna=False)
    .agg(
        reliability_mean=("reliability", "mean"),
        reliability_median=("reliability", "median"),
        reliability_q25=("reliability", lambda x: x.quantile(0.25)),
        reliability_q75=("reliability", lambda x: x.quantile(0.75)),
        sad_median=("sad", "median"),
        sad_mean=("sad", "mean"),
        sad_q25=("sad", lambda x: x.quantile(0.25)),
        sad_q75=("sad", lambda x: x.quantile(0.75)),
        edist_median=("edist", "median"),
        edist_mean=("edist", "mean"),
        outlier_fraction=("is_outlier", "mean"),
    )
    .reset_index()
)

field_summary = field_base.merge(
    field_scores,
    on=["scene_id", "field_id", "class_label"],
    how="left",
)

print("field_summary:", field_summary.shape)
display(field_summary.head())

field_summary: (190914, 22)


,scene_id,field_id,class_label,class_name,persistence_group,setting,n_pixels,class_supported,class_n_pixels,class_n_fields,class_n_scenes,reliability_mean,reliability_median,reliability_q25,reliability_q75,sad_median,sad_mean,sad_q25,sad_q75,edist_median,edist_mean,outlier_fraction
0,Scena_1,1,221,Vineyards,semi_dynamic,full,3,True,244804,27419,24,1.000000,1.000000,1.000000,1.000000,0.121537,0.119158,0.117686,0.121820,0.121462,0.119087,0.0
1,Scena_1,2,221,Vineyards,semi_dynamic,full,2,True,244804,27419,24,0.845717,0.845717,0.768576,0.922859,0.121937,0.121937,0.089304,0.154569,0.121796,0.121796,0.0
2,Scena_1,5,221,Vineyards,semi_dynamic,full,1,True,244804,27419,24,1.000000,1.000000,1.000000,1.000000,0.097143,0.097143,0.097143,0.097143,0.097106,0.097106,0.0
3,Scena_1,7,221,Vineyards,semi_dynamic,full,1,True,244804,27419,24,1.000000,1.000000,1.000000,1.000000,0.086871,0.086871,0.086871,0.086871,0.086844,0.086844,0.0
4,Scena_1,8,221,Vineyards,semi_dynamic,full,1,True,244804,27419,24,0.975407,0.975407,0.975407,0.975407,0.149283,0.149283,0.149283,0.149283,0.149144,0.149144,0.0


## 9. Audit flags

In [13]:
field_summary["audit_flag"] = "unassigned"

field_summary.loc[
    field_summary["n_pixels"] < MIN_FIELD_PIXELS,
    "audit_flag"
] = "too_small"

field_summary.loc[
    (field_summary["n_pixels"] >= MIN_FIELD_PIXELS) &
    (~field_summary["class_supported"]),
    "audit_flag"
] = "insufficient_class_support"

eligible = (
    (field_summary["n_pixels"] >= MIN_FIELD_PIXELS) &
    (field_summary["class_supported"])
)

eligible_scores = field_summary[eligible].copy()

if len(eligible_scores) == 0:
    raise ValueError("No eligible field-scene units after field/class support filtering.")

priority_reliability_thr = eligible_scores["reliability_median"].quantile(PRIORITY_RELIABILITY_Q)
uncertain_reliability_thr = eligible_scores["reliability_median"].quantile(UNCERTAIN_RELIABILITY_Q)
priority_outlier_thr = eligible_scores["outlier_fraction"].quantile(PRIORITY_OUTLIER_FRACTION_Q)

print("Thresholds:")
print(" priority_reliability_thr:", priority_reliability_thr)
print(" uncertain_reliability_thr:", uncertain_reliability_thr)
print(" priority_outlier_thr:", priority_outlier_thr)

field_summary.loc[
    eligible &
    (
        (field_summary["reliability_median"] <= priority_reliability_thr) |
        (field_summary["outlier_fraction"] >= priority_outlier_thr)
    ),
    "audit_flag"
] = "priority_for_inspection"

field_summary.loc[
    eligible &
    (field_summary["audit_flag"] == "unassigned") &
    (field_summary["reliability_median"] <= uncertain_reliability_thr),
    "audit_flag"
] = "uncertain"

field_summary.loc[
    eligible &
    (field_summary["audit_flag"] == "unassigned"),
    "audit_flag"
] = "reliable"

field_summary["priority_rank"] = np.nan

priority_rank_df = (
    field_summary[eligible]
    .sort_values(
        ["reliability_median", "outlier_fraction", "sad_median", "n_pixels"],
        ascending=[True, False, False, False],
    )
)

field_summary.loc[priority_rank_df.index, "priority_rank"] = np.arange(1, len(priority_rank_df) + 1)

print("\nAudit flags:")
print(field_summary["audit_flag"].value_counts())

display(field_summary.sort_values(["audit_flag", "priority_rank"], na_position="last").head(30))

Thresholds:
 priority_reliability_thr: 0.34423111081123353
 uncertain_reliability_thr: 0.7282259464263916
 priority_outlier_thr: 0.07142857142857142

Audit flags:
audit_flag
too_small                     150076
reliable                       27863
uncertain                       7132
priority_for_inspection         5805
insufficient_class_support        38
Name: count, dtype: int64


,scene_id,field_id,class_label,class_name,persistence_group,setting,n_pixels,class_supported,class_n_pixels,class_n_fields,class_n_scenes,reliability_mean,reliability_median,reliability_q25,reliability_q75,sad_median,sad_mean,sad_q25,sad_q75,edist_median,edist_mean,outlier_fraction,audit_flag,priority_rank
18780,Scena_10,10038,2123,Irrigated horticultural crops,dynamic_agricultural,full,13,False,610,161,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,insufficient_class_support,NaN
62893,Scena_15,4016,2123,Irrigated horticultural crops,dynamic_agricultural,full,11,False,610,161,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,insufficient_class_support,NaN
63295,Scena_15,4623,2123,Irrigated horticultural crops,dynamic_agricultural,full,13,False,610,161,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,insufficient_class_support,NaN
63296,Scena_15,4625,2123,Irrigated horticultural crops,dynamic_agricultural,full,14,False,610,161,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,insufficient_class_support,NaN
75853,Scena_16,18861,334,Burnt areas,semi_dynamic,full,19,False,154,25,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,insufficient_class_support,NaN
83197,Scena_18,4352,2123,Irrigated horticultural crops,dynamic_agricultural,full,11,False,610,161,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,insufficient_class_support,NaN
96415,Scena_2,5114,2123,Irrigated horticultural crops,dynamic_agricultural,full,19,False,610,161,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,insufficient_class_support,NaN
96652,Scena_2,5628,224,Other permanent crops,semi_dynamic,full,44,False,828,109,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,insufficient_class_support,NaN
114910,Scena_22,1074,224,Other permanent crops,semi_dynamic,full,17,False,828,109,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,insufficient_class_support,NaN
115431,Scena_22,1989,224,Other permanent crops,semi_dynamic,full,11,False,828,109,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,insufficient_class_support,NaN


## 10. Class-level summary

In [14]:
class_summary = (
    field_summary[field_summary["class_supported"]]
    .groupby(["class_label", "class_name", "persistence_group", "setting"], dropna=False)
    .agg(
        n_pixels=("n_pixels", "sum"),
        n_scenes=("scene_id", "nunique"),
        n_fields=("field_id", "count"),
        sad_median=("sad_median", "median"),
        sad_mean=("sad_mean", "mean"),
        sad_q25=("sad_q25", "median"),
        sad_q75=("sad_q75", "median"),
        edist_median=("edist_median", "median"),
        edist_mean=("edist_mean", "mean"),
        reliability_mean=("reliability_mean", "mean"),
        reliability_median=("reliability_median", "median"),
        reliability_q25=("reliability_q25", "median"),
        reliability_q75=("reliability_q75", "median"),
        outlier_fraction=("outlier_fraction", "mean"),
    )
    .reset_index()
)

display(class_summary.sort_values("reliability_median"))

,class_label,class_name,persistence_group,setting,n_pixels,n_scenes,n_fields,sad_median,sad_mean,sad_q25,sad_q75,edist_median,edist_mean,reliability_mean,reliability_median,reliability_q25,reliability_q75,outlier_fraction
8,312,Coniferous forest,persistent,full,28387,24,1403,0.146484,0.176969,0.119632,0.183723,0.146336,0.176408,0.697322,0.827612,0.645293,0.954591,0.010640
9,313,Mixed forest,persistent,full,23916,23,1309,0.138359,0.162528,0.111384,0.167789,0.138249,0.162139,0.693453,0.828364,0.638284,1.000000,0.056635
7,311,Broad-leaved forest,persistent,full,138395,25,4471,0.166341,0.190565,0.136994,0.198526,0.166150,0.189970,0.687475,0.837595,0.649298,0.977737,0.053639
15,332,Bare rock and outcrops,persistent,full,2500,21,285,0.124249,0.162396,0.108525,0.148272,0.124170,0.161547,0.688519,0.862160,0.676734,1.000000,0.110704
16,333,Sparsely vegetated areas,semi_dynamic,full,6517,23,275,0.120348,0.142879,0.104371,0.146573,0.120275,0.142646,0.733336,0.929311,0.703409,1.000000,0.058324
10,314,Other wooded / forest class,persistent,full,17369,24,2058,0.137402,0.154285,0.121210,0.153733,0.137294,0.154002,0.759930,0.930965,0.814030,1.000000,0.010485
4,241,Annual crops associated with permanent crops,dynamic_agricultural,full,25210,24,5124,0.091702,0.122050,0.083332,0.101764,0.091670,0.121813,0.737120,0.947891,0.844451,1.000000,0.093103
12,322,Shrubland,persistent,full,33325,25,4528,0.154671,0.170763,0.136490,0.174821,0.154505,0.170408,0.783500,0.957686,0.835522,1.000000,0.003171
2,223,Olive groves,persistent,full,734113,25,44931,0.113618,0.140277,0.098810,0.131245,0.113557,0.139995,0.761694,0.961607,0.834123,1.000000,0.032446
20,3241,Natural recolonization areas,semi_dynamic,full,7423,23,922,0.149999,0.162341,0.132295,0.166417,0.149858,0.162019,0.799983,0.964415,0.865052,1.000000,0.006284


## 11. Diagnostics

In [15]:
print("\nAudit flags:")
print(field_summary["audit_flag"].value_counts())

print("\nClass support summary:")
print(class_support["class_supported"].value_counts())

P = field_summary[field_summary["audit_flag"] == "priority_for_inspection"].copy()

print("\nPriority fields by scene:")
print(P["scene_id"].value_counts())

print("\nPriority fields by original class:")
print(P["class_label"].value_counts().sort_index())

print("\nUnsupported classes:")
display(class_support[~class_support["class_supported"]].sort_values("class_n_pixels", ascending=False))

print("\nTop priority examples:")
display(
    P.sort_values("priority_rank")
    .head(30)
    [
        [
            "priority_rank", "scene_id", "field_id", "class_label", "class_name",
            "n_pixels", "reliability_median", "outlier_fraction", "sad_median",
            "audit_flag"
        ]
    ]
)


Audit flags:
audit_flag
too_small                     150076
reliable                       27863
uncertain                       7132
priority_for_inspection         5805
insufficient_class_support        38
Name: count, dtype: int64

Class support summary:
class_supported
True     22
False     4
Name: count, dtype: int64

Priority fields by scene:
scene_id
Scena_9     949
Scena_20    743
Scena_21    502
Scena_25    481
Scena_11    458
Scena_24    431
Scena_5     354
Scena_2     227
Scena_10    221
Scena_23    217
Scena_18    196
Scena_19    162
Scena_6     161
Scena_22    120
Scena_14    113
Scena_4     102
Scena_1      94
Scena_13     89
Scena_3      49
Scena_16     40
Scena_12     36
Scena_17     17
Scena_8      16
Scena_15     14
Scena_7      13
Name: count, dtype: int64

Priority fields by original class:
class_label
221      644
222      286
223     1952
231       11
241      100
242        9
243        2
311      190
312       58
313       85
314       33
321      396
322     

,class_label,class_n_pixels,class_n_fields,class_n_scenes,class_supported,class_name,persistence_group
3,224,828,109,10,False,Other permanent crops,semi_dynamic
23,2123,610,161,21,False,Irrigated horticultural crops,dynamic_agricultural
8,244,257,60,3,False,Agro-forestry areas,semi_dynamic
19,334,154,25,4,False,Burnt areas,semi_dynamic



Top priority examples:


,priority_rank,scene_id,field_id,class_label,class_name,n_pixels,reliability_median,outlier_fraction,sad_median,audit_flag
92782,1.0,Scena_19,10621,241,Annual crops associated with permanent crops,10,0.004542,0.900000,0.495760,priority_for_inspection
107281,2.0,Scena_21,682,241,Annual crops associated with permanent crops,15,0.004777,1.000000,0.490640,priority_for_inspection
161183,3.0,Scena_6,1582,241,Annual crops associated with permanent crops,10,0.005244,1.000000,0.483829,priority_for_inspection
162175,4.0,Scena_6,3385,241,Annual crops associated with permanent crops,18,0.007486,1.000000,0.457016,priority_for_inspection
102503,5.0,Scena_20,6764,241,Annual crops associated with permanent crops,20,0.008037,0.900000,0.451811,priority_for_inspection
27361,6.0,Scena_11,9878,241,Annual crops associated with permanent crops,13,0.008363,1.000000,0.448709,priority_for_inspection
107647,7.0,Scena_21,1218,241,Annual crops associated with permanent crops,22,0.008541,1.000000,0.447136,priority_for_inspection
189220,8.0,Scena_9,8879,241,Annual crops associated with permanent crops,10,0.010135,1.000000,0.434909,priority_for_inspection
96843,9.0,Scena_2,5971,241,Annual crops associated with permanent crops,19,0.010675,1.000000,0.430437,priority_for_inspection
105243,10.0,Scena_20,11194,241,Annual crops associated with permanent crops,15,0.013468,1.000000,0.413031,priority_for_inspection


## 12. Save outputs

In [16]:
field_summary_path = TABLE_DIR / f"field_spectral_reliability_{SETTING}.csv"
class_summary_path = TABLE_DIR / f"class_spectral_reliability_summary_{SETTING}.csv"
class_support_path = TABLE_DIR / f"class_support_{SETTING}.csv"
pixel_scores_path = TABLE_DIR / f"pixel_spectral_reliability_{SETTING}.csv"

field_summary.to_csv(field_summary_path, index=False)
class_summary.to_csv(class_summary_path, index=False)
class_support.to_csv(class_support_path, index=False)
df_scores.to_csv(pixel_scores_path, index=False)

field_summary.to_csv(TABLE_DIR / "field_reliability_scores_ORIGINAL_CLASSES.csv", index=False)
class_summary.to_csv(TABLE_DIR / "class_reliability_summary_ORIGINAL_CLASSES.csv", index=False)
class_support.to_csv(TABLE_DIR / "class_support_ORIGINAL_CLASSES.csv", index=False)

print("Saved:")
print(" ", field_summary_path)
print(" ", class_summary_path)
print(" ", class_support_path)
print(" ", pixel_scores_path)
print(" ", TABLE_DIR / "field_reliability_scores_ORIGINAL_CLASSES.csv")
print(" ", TABLE_DIR / "class_reliability_summary_ORIGINAL_CLASSES.csv")
print(" ", TABLE_DIR / "class_support_ORIGINAL_CLASSES.csv")

Saved:
  /kaggle/working/tables/field_spectral_reliability_full.csv
  /kaggle/working/tables/class_spectral_reliability_summary_full.csv
  /kaggle/working/tables/class_support_full.csv
  /kaggle/working/tables/pixel_spectral_reliability_full.csv
  /kaggle/working/tables/field_reliability_scores_ORIGINAL_CLASSES.csv
  /kaggle/working/tables/class_reliability_summary_ORIGINAL_CLASSES.csv
  /kaggle/working/tables/class_support_ORIGINAL_CLASSES.csv


In [19]:
# =========================================================
# Complete SIT LIVELLO_4 class inventory and appendix legend
# =========================================================

import pandas as pd
from pathlib import Path

TABLE_DIR = Path(TABLE_DIR)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

required_cols = [
    "class_label",
    "class_name",
    "persistence_group",
    "class_n_pixels",
    "class_n_fields",
    "class_n_scenes",
    "class_supported",
]

missing = [c for c in required_cols if c not in class_support.columns]
if missing:
    raise ValueError(f"Missing required columns in class_support: {missing}")

short_label_map = {
    2111: "Non-ir. arable",
    2112: "Non-ir. hortic.",
    2121: "Irr. arable",
    2123: "Rice fields",
    221:  "Vineyards",
    222:  "Fruit trees",
    223:  "Olive groves",
    224:  "Other perm. crops",
    231:  "Pastures",
    241:  "Ann./perm. crops",
    242:  "Complex cultiv.",
    243:  "Agric./natural veg.",
    244:  "Agro-forestry",
    311:  "Broadleaf forest",
    312:  "Coniferous forest",
    313:  "Mixed forest",
    314:  "Wooded past./meadows",
    321:  "Grass/pasture",
    322:  "Shrubland",
    323:  "Scleroph. veg.",
    3241: "Recolonization",
    3242: "Young reforestation",
    331:  "Beaches/dunes",
    332:  "Bare rock",
    333:  "Sparse vegetation",
    334:  "Burnt areas",
}

class_inventory = class_support.copy()

class_inventory["class_label"] = class_inventory["class_label"].astype(int)
class_inventory["short_label"] = (
    class_inventory["class_label"]
    .map(short_label_map)
    .fillna(class_inventory["class_name"])
)

class_inventory["prototype_support"] = class_inventory["class_supported"].map({
    True: "Yes",
    False: "No",
})

class_inventory["support_note"] = class_inventory["class_supported"].map({
    True: "used for prototype estimation",
    False: "insufficient_class_support",
})

class_inventory = class_inventory.rename(columns={
    "class_label": "code",
    "class_name": "official_sit_label",
    "persistence_group": "group",
    "class_n_pixels": "pixels",
    "class_n_fields": "field_scene_units",
    "class_n_scenes": "scenes",
})

class_inventory = class_inventory[
    [
        "code",
        "official_sit_label",
        "short_label",
        "group",
        "pixels",
        "field_scene_units",
        "scenes",
        "prototype_support",
        "support_note",
    ]
].sort_values("code").reset_index(drop=True)

class_inventory_path = TABLE_DIR / "class_inventory_full.csv"
class_inventory.to_csv(class_inventory_path, index=False)

print("Saved:", class_inventory_path)
display(class_inventory)

Saved: tables/class_inventory_full.csv


,code,official_sit_label,short_label,group,pixels,field_scene_units,scenes,prototype_support,support_note
0,221,Vineyards,Vineyards,semi_dynamic,244804,27419,24,Yes,used for prototype estimation
1,222,Fruit trees and minor fruit plantations,Fruit trees,semi_dynamic,117809,13277,25,Yes,used for prototype estimation
2,223,Olive groves,Olive groves,persistent,734113,44931,25,Yes,used for prototype estimation
3,224,Other permanent crops,Other perm. crops,semi_dynamic,828,109,10,No,insufficient_class_support
4,231,Dense herbaceous cover,Pastures,semi_dynamic,3693,597,17,Yes,used for prototype estimation
5,241,Annual crops associated with permanent crops,Ann./perm. crops,dynamic_agricultural,25210,5124,24,Yes,used for prototype estimation
6,242,Complex cultivation patterns,Complex cultiv.,dynamic_agricultural,3785,1430,24,Yes,used for prototype estimation
7,243,Agricultural areas with natural vegetation,Agric./natural veg.,semi_dynamic,1719,435,21,Yes,used for prototype estimation
8,244,Agro-forestry areas,Agro-forestry,semi_dynamic,257,60,3,No,insufficient_class_support
9,311,Broad-leaved forest,Broadleaf forest,persistent,138395,4471,25,Yes,used for prototype estimation


In [20]:
# =========================================================
# Export complete SIT legend as LaTeX longtable
# =========================================================

def latex_escape(x):
    x = str(x)
    replacements = {
        "&": r"\&",
        "%": r"\%",
        "_": r"\_",
        "#": r"\#",
    }
    for old, new in replacements.items():
        x = x.replace(old, new)
    return x

rows = []

for _, r in class_inventory.iterrows():
    rows.append(
        f'{int(r["code"])} & '
        f'{latex_escape(r["official_sit_label"])} & '
        f'{latex_escape(r["short_label"])} & '
        f'{latex_escape(r["group"])} & '
        f'{int(r["pixels"])} & '
        f'{int(r["field_scene_units"])} & '
        f'{int(r["scenes"])} & '
        f'{latex_escape(r["prototype_support"])} \\\\'
    )

latex_rows = "\n".join(rows)

latex_table = rf"""
\begin{{longtable}}{{r p{{0.31\textwidth}} p{{0.18\textwidth}} p{{0.13\textwidth}} r r r c}}
\caption{{Complete legend of the original SIT LIVELLO\_4 classes represented in
the exported PRISMA--SIT inventory.}}
\label{{tab:complete_sit_legend}}\\
\toprule
Code & Official SIT label & Short label & Group & Pixels & FS units & Scenes & Prot. \\
\midrule
\endfirsthead

\toprule
Code & Official SIT label & Short label & Group & Pixels & FS units & Scenes & Prot. \\
\midrule
\endhead

{latex_rows}

\bottomrule
\end{{longtable}}
""".strip()

latex_path = TABLE_DIR / "tab_complete_sit_legend_full.tex"
latex_path.write_text(latex_table)

print("Saved:", latex_path)
print(latex_table)

Saved: tables/tab_complete_sit_legend_full.tex
\begin{longtable}{r p{0.31\textwidth} p{0.18\textwidth} p{0.13\textwidth} r r r c}
\caption{Complete legend of the original SIT LIVELLO\_4 classes represented in
the exported PRISMA--SIT inventory.}
\label{tab:complete_sit_legend}\\
\toprule
Code & Official SIT label & Short label & Group & Pixels & FS units & Scenes & Prot. \\
\midrule
\endfirsthead

\toprule
Code & Official SIT label & Short label & Group & Pixels & FS units & Scenes & Prot. \\
\midrule
\endhead

221 & Vineyards & Vineyards & semi\_dynamic & 244804 & 27419 & 24 & Yes \\
222 & Fruit trees and minor fruit plantations & Fruit trees & semi\_dynamic & 117809 & 13277 & 25 & Yes \\
223 & Olive groves & Olive groves & persistent & 734113 & 44931 & 25 & Yes \\
224 & Other permanent crops & Other perm. crops & semi\_dynamic & 828 & 109 & 10 & No \\
231 & Dense herbaceous cover & Pastures & semi\_dynamic & 3693 & 597 & 17 & Yes \\
241 & Annual crops associated with permanent crops 

## 13. Scene priority-rate diagnostic

In [17]:
eligible_fields = field_summary[
    (field_summary["n_pixels"] >= MIN_FIELD_PIXELS) &
    (field_summary["class_supported"])
].copy()

global_priority_rate = eligible_fields["audit_flag"].eq("priority_for_inspection").mean()

scene_priority = (
    eligible_fields
    .groupby("scene_id")
    .agg(
        n_eligible=("field_id", "count"),
        n_priority=("audit_flag", lambda x: (x == "priority_for_inspection").sum()),
        priority_rate=("audit_flag", lambda x: (x == "priority_for_inspection").mean()),
        reliability_median=("reliability_median", "median"),
        outlier_fraction_median=("outlier_fraction", "median"),
    )
    .reset_index()
)

scene_priority["priority_rate_pct"] = 100 * scene_priority["priority_rate"]
scene_priority["above_global"] = scene_priority["priority_rate"] > global_priority_rate

print(f"Global priority rate: {100 * global_priority_rate:.2f}%")
display(scene_priority.sort_values("priority_rate_pct", ascending=False))

scene_priority.to_csv(TABLE_DIR / f"scene_priority_rates_{SETTING}.csv", index=False)
print("Saved:", TABLE_DIR / f"scene_priority_rates_{SETTING}.csv")

Global priority rate: 14.23%


,scene_id,n_eligible,n_priority,priority_rate,reliability_median,outlier_fraction_median,priority_rate_pct,above_global
12,Scena_20,1716,743,0.432984,0.488856,0.0,43.298368,True
24,Scena_9,2209,949,0.429606,0.385629,0.0,42.960616,True
20,Scena_5,987,354,0.358663,0.770262,0.0,35.866261,True
16,Scena_24,1415,431,0.304594,0.649325,0.0,30.459364,True
13,Scena_21,1714,502,0.292882,0.645404,0.0,29.288215,True
2,Scena_11,1922,458,0.238293,0.953657,0.0,23.829344,True
21,Scena_6,710,161,0.226761,0.997900,0.0,22.676056,True
17,Scena_25,2178,481,0.220845,0.835417,0.0,22.084481,True
11,Scena_2,1110,227,0.204505,0.900810,0.0,20.450450,True
9,Scena_18,984,196,0.199187,0.897779,0.0,19.918699,True


Saved: /kaggle/working/tables/scene_priority_rates_full.csv


## 14. Threshold sensitivity analysis

This diagnostic evaluates whether the spectrally-atypical field--scene set is stable under nearby operating quantile thresholds. It uses the already computed field-level support score (`reliability_median`, i.e. the field median of the pixel-level spectral support score) and `outlier_fraction`. The analysis is not used to optimize semantic-error detection; it only assesses the stability of the diagnostic stratification.


In [18]:
# =========================================================
# THRESHOLD SENSITIVITY ANALYSIS
# =========================================================

# This cell compares the main operating point with a more conservative
# and a more permissive threshold configuration.
#
# Main rule:
#   SA if S_f <= q_low(S_f) OR o_f >= q_out(o_f)
#   IS if q_low(S_f) < S_f <= q_mid(S_f) AND o_f < q_out(o_f)
#   ST otherwise
#
# Here S_f is the field-level spectral support score. In the current notebook
# it is stored as `reliability_median` for backward compatibility.

S_COL = "reliability_median"
O_COL = "outlier_fraction"
ID_COLS = ["scene_id", "field_id", "class_label"]

required_cols = ID_COLS + [S_COL, O_COL, "n_pixels", "class_supported"]
missing_cols = [c for c in required_cols if c not in field_summary.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in field_summary: {missing_cols}")

threshold_eligible = field_summary[
    (field_summary["n_pixels"] >= MIN_FIELD_PIXELS) &
    (field_summary["class_supported"]) &
    field_summary[S_COL].notna() &
    field_summary[O_COL].notna()
].copy()

if len(threshold_eligible) == 0:
    raise ValueError("No eligible field-scene units available for threshold sensitivity analysis.")

print("Eligible field-scene units for threshold sensitivity:", len(threshold_eligible))
print("Support score range:", float(threshold_eligible[S_COL].min()), float(threshold_eligible[S_COL].max()))
print("Outlier fraction range:", float(threshold_eligible[O_COL].min()), float(threshold_eligible[O_COL].max()))


def assign_strata_by_quantiles(df, q_low=0.10, q_mid=0.30, q_out=0.90):
    """Assign SA/IS/ST strata using quantiles of S_f and o_f."""
    out = df.copy()

    tau_low = out[S_COL].quantile(q_low)
    tau_mid = out[S_COL].quantile(q_mid)
    omega = out[O_COL].quantile(q_out)

    is_sa = (out[S_COL] <= tau_low) | (out[O_COL] >= omega)
    is_is = (
        (out[S_COL] > tau_low) &
        (out[S_COL] <= tau_mid) &
        (out[O_COL] < omega)
    )

    out["stratum_tmp"] = "spectrally_typical"
    out.loc[is_is, "stratum_tmp"] = "intermediate_support"
    out.loc[is_sa, "stratum_tmp"] = "spectrally_atypical"

    thresholds = {
        "q_low": q_low,
        "q_mid": q_mid,
        "q_out": q_out,
        "tau_low": tau_low,
        "tau_mid": tau_mid,
        "omega": omega,
    }
    return out, thresholds


def field_keys(df, mask):
    return set(df.loc[mask, ID_COLS].apply(tuple, axis=1))


def scene_sa_rates(df):
    return (
        df
        .assign(is_sa=df["stratum_tmp"].eq("spectrally_atypical"))
        .groupby("scene_id")
        .agg(
            eligible=("is_sa", "size"),
            n_sa=("is_sa", "sum"),
        )
        .reset_index()
        .assign(sa_rate=lambda x: x["n_sa"] / x["eligible"])
    )


threshold_configs = [
    {"thresholds": "5/25/95",  "q_low": 0.05, "q_mid": 0.25, "q_out": 0.95},
    {"thresholds": "10/30/90", "q_low": 0.10, "q_mid": 0.30, "q_out": 0.90},
    {"thresholds": "15/35/85", "q_low": 0.15, "q_mid": 0.35, "q_out": 0.85},
]

# Main configuration for set and scene-rate comparisons.
main_cfg = {"q_low": PRIORITY_RELIABILITY_Q, "q_mid": UNCERTAIN_RELIABILITY_Q, "q_out": PRIORITY_OUTLIER_FRACTION_Q}
main_df, main_thresholds = assign_strata_by_quantiles(threshold_eligible, **main_cfg)
main_sa_keys = field_keys(main_df, main_df["stratum_tmp"].eq("spectrally_atypical"))
main_scene = scene_sa_rates(main_df)[["scene_id", "sa_rate"]].rename(columns={"sa_rate": "sa_rate_main"})

total_eligible = len(threshold_eligible)
rows = []
scene_rows = []

for cfg in threshold_configs:
    tmp, thr = assign_strata_by_quantiles(
        threshold_eligible,
        q_low=cfg["q_low"],
        q_mid=cfg["q_mid"],
        q_out=cfg["q_out"],
    )

    sa_keys = field_keys(tmp, tmp["stratum_tmp"].eq("spectrally_atypical"))
    intersection = len(main_sa_keys & sa_keys)
    union = len(main_sa_keys | sa_keys)

    jaccard = intersection / union if union > 0 else np.nan
    recall_main = intersection / len(main_sa_keys) if len(main_sa_keys) > 0 else np.nan

    n_sa = len(sa_keys)
    rows.append({
        "thresholds": cfg["thresholds"],
        "q_low": cfg["q_low"],
        "q_mid": cfg["q_mid"],
        "q_out": cfg["q_out"],
        "n_eligible": total_eligible,
        "n_sa": n_sa,
        "sa_rate": n_sa / total_eligible,
        "jaccard_vs_main": jaccard,
        "recall_main_sa": recall_main,
        "tau_low": thr["tau_low"],
        "tau_mid": thr["tau_mid"],
        "omega": thr["omega"],
    })

    alt_scene = scene_sa_rates(tmp)[["scene_id", "sa_rate"]].rename(columns={"sa_rate": "sa_rate_alt"})
    merged = main_scene.merge(alt_scene, on="scene_id", how="inner")

    scene_spearman = merged["sa_rate_main"].corr(merged["sa_rate_alt"], method="spearman")

    top_main = set(merged.sort_values("sa_rate_main", ascending=False).head(10)["scene_id"])
    top_alt = set(merged.sort_values("sa_rate_alt", ascending=False).head(10)["scene_id"])
    top10_scene_jaccard = len(top_main & top_alt) / len(top_main | top_alt) if (top_main | top_alt) else np.nan

    scene_rows.append({
        "thresholds": cfg["thresholds"],
        "scene_spearman": scene_spearman,
        "top10_scene_jaccard": top10_scene_jaccard,
    })

threshold_sensitivity = pd.DataFrame(rows)
scene_threshold_sensitivity = pd.DataFrame(scene_rows)
threshold_sensitivity_final = threshold_sensitivity.merge(
    scene_threshold_sensitivity,
    on="thresholds",
    how="left",
)

threshold_sensitivity_final["sa_rate_pct"] = 100 * threshold_sensitivity_final["sa_rate"]
threshold_sensitivity_final["jaccard_vs_main_pct"] = 100 * threshold_sensitivity_final["jaccard_vs_main"]
threshold_sensitivity_final["recall_main_sa_pct"] = 100 * threshold_sensitivity_final["recall_main_sa"]
threshold_sensitivity_final["top10_scene_jaccard_pct"] = 100 * threshold_sensitivity_final["top10_scene_jaccard"]

threshold_sensitivity_path = TABLE_DIR / f"threshold_sensitivity_summary_{SETTING}.csv"
threshold_sensitivity_final.to_csv(threshold_sensitivity_path, index=False)

print("Saved:", threshold_sensitivity_path)
display(
    threshold_sensitivity_final[[
        "thresholds", "n_eligible", "n_sa", "sa_rate_pct",
        "jaccard_vs_main_pct", "recall_main_sa_pct",
        "scene_spearman", "top10_scene_jaccard_pct",
    ]]
)



Eligible field-scene units for threshold sensitivity: 40800
Support score range: 0.004541559144854546 1.0
Outlier fraction range: 0.0 1.0
Saved: /kaggle/working/tables/threshold_sensitivity_summary_full.csv


,thresholds,n_eligible,n_sa,sa_rate_pct,jaccard_vs_main_pct,recall_main_sa_pct,scene_spearman,top10_scene_jaccard_pct
0,5/25/95,40800,2698,6.612745,46.477175,46.477175,0.986154,81.818182
1,10/30/90,40800,5805,14.227941,100.000000,100.000000,1.000000,100.000000
2,15/35/85,40800,8889,21.786765,65.305434,100.000000,0.983077,100.000000


In [19]:
# =========================================================
# EXPORT COMPACT LATEX TABLE FOR THE MANUSCRIPT APPENDIX
# =========================================================

latex_rows = []
for _, r in threshold_sensitivity_final.iterrows():
    latex_rows.append(
        f"{r['thresholds']} & "
        f"{int(r['n_sa']):,} & "
        f"{r['sa_rate_pct']:.1f} & "
        f"{r['jaccard_vs_main_pct']:.1f} & "
        f"{r['recall_main_sa_pct']:.1f} & "
        f"{r['scene_spearman']:.2f} \\\\"
    )

latex = r"""\begin{table}[t]
\centering
\scriptsize
\caption{Sensitivity of the spectral-atypicality stratum to the operating quantile thresholds.}
\label{tab:sensitivity_thresholds}
\begin{tabular}{lrrrrr}
\toprule
Thresholds & SA & Rate & Jac. & Rec. & $\rho_{\mathrm{scene}}$ \\
\midrule
""" + "\n".join(latex_rows) + r"""
\bottomrule
\end{tabular}

\vspace{1mm}
\footnotesize
Thresholds indicate $(q_S^{low}, q_S^{mid}, q_o)$ for the field-level support
score and outlier fraction. Jac. and Rec. are computed relative to the main
10/30/90 configuration. $\rho_{\mathrm{scene}}$ is the Spearman correlation of
scene-level SA rates with the main configuration.
\end{table}
"""

threshold_sensitivity_tex_path = TABLE_DIR / f"tab_sensitivity_thresholds_{SETTING}.tex"
threshold_sensitivity_tex_path.write_text(latex)

print(latex)
print("Saved:", threshold_sensitivity_tex_path)



\begin{table}[t]
\centering
\scriptsize
\caption{Sensitivity of the spectral-atypicality stratum to the operating quantile thresholds.}
\label{tab:sensitivity_thresholds}
\begin{tabular}{lrrrrr}
\toprule
Thresholds & SA & Rate & Jac. & Rec. & $\rho_{\mathrm{scene}}$ \\
\midrule
5/25/95 & 2,698 & 6.6 & 46.5 & 46.5 & 0.99 \\
10/30/90 & 5,805 & 14.2 & 100.0 & 100.0 & 1.00 \\
15/35/85 & 8,889 & 21.8 & 65.3 & 100.0 & 0.98 \\
\bottomrule
\end{tabular}

\vspace{1mm}
\footnotesize
Thresholds indicate $(q_S^{low}, q_S^{mid}, q_o)$ for the field-level support
score and outlier fraction. Jac. and Rec. are computed relative to the main
10/30/90 configuration. $\rho_{\mathrm{scene}}$ is the Spearman correlation of
scene-level SA rates with the main configuration.
\end{table}

Saved: /kaggle/working/tables/tab_sensitivity_thresholds_full.tex


In [20]:
from pathlib import Path
import shutil
import os

SOURCE_DIR = Path("/kaggle/working")
ZIP_BASE = Path("/kaggle/working/kaggle_working_backup")

# Avoid including a previous version of the same zip inside the new zip
zip_path = Path(str(ZIP_BASE) + ".zip")
if zip_path.exists():
    zip_path.unlink()

# Optional: remove older backup zips to avoid nesting large files
for old_zip in SOURCE_DIR.glob("kaggle_working_backup*.zip"):
    print("Removing old zip:", old_zip)
    old_zip.unlink()

print("Compressing:", SOURCE_DIR)
print("Output:", zip_path)

shutil.make_archive(
    base_name=str(ZIP_BASE),
    format="zip",
    root_dir=SOURCE_DIR
)

print("Done.")
print("ZIP file:", zip_path)
print("Size MB:", zip_path.stat().st_size / 1e6)


from IPython.display import FileLink

# Assicurati di essere nella cartella giusta (di solito /kaggle/working)
import os
os.chdir('/kaggle/working')

# Crea il link per il tuo file (cambia 'nome_del_tuo_file.zip')
FileLink('kaggle_working_backup.zip')

Compressing: /kaggle/working
Output: /kaggle/working/kaggle_working_backup.zip
Done.
ZIP file: /kaggle/working/kaggle_working_backup.zip
Size MB: 94.359507


/kaggle/working/kaggle_working_backup.zip

In [21]:
scene_priority = pd.read_csv(TABLE_DIR / "scene_priority_rates_full.csv")
print(
    scene_priority
    .sort_values("priority_rate_pct", ascending=False)
    .to_string(index=False)
)

scene_id  n_eligible  n_priority  priority_rate  reliability_median  outlier_fraction_median  priority_rate_pct  above_global
Scena_20        1716         743       0.432984            0.488856                      0.0          43.298368          True
 Scena_9        2209         949       0.429606            0.385629                      0.0          42.960616          True
 Scena_5         987         354       0.358663            0.770262                      0.0          35.866261          True
Scena_24        1415         431       0.304594            0.649325                      0.0          30.459364          True
Scena_21        1714         502       0.292882            0.645404                      0.0          29.288215          True
Scena_11        1922         458       0.238293            0.953657                      0.0          23.829344          True
 Scena_6         710         161       0.226761            0.997900                      0.0          22.676056       

In [22]:
class_summary = pd.read_csv(TABLE_DIR / "class_spectral_reliability_summary_full.csv")

print(
    class_summary[
        [
            "class_label",
            "class_name",
            "persistence_group",
            "n_pixels",
            "n_scenes",
            "n_fields",
            "reliability_median",
            "sad_median",
            "outlier_fraction",
        ]
    ]
    .sort_values("reliability_median")
    .to_string(index=False)
)

 class_label                                   class_name    persistence_group  n_pixels  n_scenes  n_fields  reliability_median  sad_median  outlier_fraction
         312                            Coniferous forest           persistent     28387        24      1403            0.827612    0.146484          0.010640
         313                                 Mixed forest           persistent     23916        23      1309            0.828364    0.138359          0.056635
         311                          Broad-leaved forest           persistent    138395        25      4471            0.837595    0.166341          0.053639
         332                       Bare rock and outcrops           persistent      2500        21       285            0.862160    0.124249          0.110704
         333                     Sparsely vegetated areas         semi_dynamic      6517        23       275            0.929311    0.120348          0.058324
         314                  Other wooded / f

In [23]:
F = pd.read_csv(TABLE_DIR / "field_spectral_reliability_full.csv")

P = F[
    (F["audit_flag"] == "priority_for_inspection") &
    (F["n_pixels"] >= 30)
].copy()

print("Priority fields total:", (F["audit_flag"] == "priority_for_inspection").sum())
print("Priority fields with n_pixels >= 30:", len(P))

print(P["scene_id"].value_counts())
print(P["class_label"].value_counts().sort_index())

Priority fields total: 5805
Priority fields with n_pixels >= 30: 1970
scene_id
Scena_20    315
Scena_9     298
Scena_21    209
Scena_5     165
Scena_25    138
Scena_11    136
Scena_24    128
Scena_2     106
Scena_10     70
Scena_18     61
Scena_22     54
Scena_23     54
Scena_19     51
Scena_6      47
Scena_1      26
Scena_14     25
Scena_4      21
Scena_3      17
Scena_16     12
Scena_13     11
Scena_12      8
Scena_17      8
Scena_15      5
Scena_7       3
Scena_8       2
Name: count, dtype: int64
class_label
221     181
222      97
223     743
231       1
241      18
311      64
312      23
313      33
314      11
321      90
322       9
323      23
331       4
332       4
333       3
2111    630
2121     32
3241      4
Name: count, dtype: int64
